# US Arb Freight Breakevens vs Spot Freight Rates

This script is used to compare the US Arb Freight Breakevens with Spark30S Spot Freight Rates.

For a full explanation of how to import our Arb Breakevens or Spark30S data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html



### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

In [ ]:
# Define the function for listing all netbacks
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    print(">>>> All the routes you can fetch")
    tickers = []
    fobPort_names = []

    availablevia = []

    for contract in content["data"]["staticData"]["fobPorts"]:
        tickers.append(contract["uuid"])
        fobPort_names.append(contract["name"])

        availablevia.append(contract["availableViaPoints"])

    reldates = content["data"]["staticData"]["sparkReleases"]

    dicto1 = content["data"]

    return tickers, fobPort_names, availablevia, reldates, dicto1

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/introduction.html

In [ ]:
# Input the path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")


# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Fetching Netbacks Arb Breakevens Data

In [ ]:
# Define the function for listing all netback reference data
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# Define the function for fetching breakevens data
def fetch_breakevens(access_token, ticker, via=None, breakeven='freight', start=None, end=None, format='json'):


    query_params = breakeven + '/' + "?fob-port={}".format(ticker)

    if via is not None:
        query_params += "&via-point={}".format(via)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)

    uri = "/v1.0/netbacks/arb-breakevens/{}".format(query_params)
    print(uri)
    content = do_api_get_query(
        uri="/v1.0/netbacks/arb-breakevens/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        my_dict = content['data']
    else:
        my_dict = content.decode('utf-8')
        my_dict = pd.read_csv(StringIO(my_dict))

    return my_dict


# Calling reference data and formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df.head(3)

In [ ]:
# Here we define which port we want using reference data
tsab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

In [ ]:
# Fetching breakevens data in CSV format

## Calling data a
break_df = fetch_breakevens(access_token, tsab, via='cogh', breakeven='freight', format='csv')

break_df['ReleaseDate'] = pd.to_datetime(break_df['ReleaseDate'])

In [ ]:
# storing length of dataset to filter Spark30S data
freight_length = len(break_df['ReleaseDate'].unique())

# 3. Fetching Spot Freight Prices

In [ ]:
# Defining function to fetch historical freight data from API
def fetch_historical_price_releases(access_token, ticker, limit, offset=None, vessel=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)
    
    if vessel is not None:
        query_params += "&vessel-type={}".format(vessel)

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params)
    print(uri)

    my_dict = content["data"]

    return my_dict


# Defining the function for storing and formatting the data into a Pandas DataFrame
def store_and_format(dict_hist):
    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "USDperday": [],
        "USDperdayMax": [],
        "USDperdayMin": [],
    }

    for release in dict_hist:

        data_points = release["data"][0]["dataPoints"]

        for data_point in data_points:
            stored_data['Ticker'].append(release["contractId"])
            stored_data['ReleaseDate'].append(release["releaseDate"])

            period_start_at = data_point["deliveryPeriod"]["startAt"]
            stored_data['PeriodStart'].append(period_start_at)

            stored_data['USDperday'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
            stored_data['USDperdayMin'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMin"])
            stored_data['USDperdayMax'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMax"])
            
        historical_df = pd.DataFrame(stored_data)
        
        historical_df["USDperday"] = pd.to_numeric(historical_df["USDperday"])
        historical_df["USDperdayMax"] = pd.to_numeric(historical_df["USDperdayMax"])
        historical_df["USDperdayMin"] = pd.to_numeric(historical_df["USDperdayMin"])

        historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
        
    return historical_df

In [ ]:
### Calling Spark30S data using above functions
spark30s_raw = fetch_historical_price_releases(access_token, 'spark30s', limit=freight_length, offset=0)
spark30s = store_and_format(spark30s_raw)

spark30s.head(5)

# 4. Plotting

## 4.1. No Formatting

In [ ]:
# fetch front month breakevens only
front_df = break_df[break_df['LoadMonthIndex']=="M+1"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(15,7))

ax.plot(spark30s['ReleaseDate'],spark30s['USDperday'], color = '#48C38D', linewidth=2.5, label='Spark30S (Atlantic)')
ax.plot(front_df['ReleaseDate'],front_df['FreightBreakeven'], color='#4F41F4', linewidth=2, label='US Arb [M+1] Freight Breakeven Level')

ax.set_xlim(datetime.datetime.today() - datetime.timedelta(days=380), datetime.datetime.today())
ax.set_ylim(-100000, 120000)

plt.title('Spark30S (Atlantic) vs. US Arb [M+1] Freight Breakeven Level')

sns.despine(left=True, bottom=True)
plt.grid(True)
plt.legend()

## 4.2. Plotting with Conditional Shading

In [ ]:
spark30s['ReleaseDate'] = pd.to_datetime(spark30s['ReleaseDate'])
merge_df = pd.merge(spark30s, front_df, on='ReleaseDate', how='inner')

In [ ]:
fig2, ax2 = plt.subplots(figsize=(15,7))

ax2.plot(merge_df['ReleaseDate'],merge_df['USDperday'], color = '#48C38D', linewidth=2.5, label='Spark30S (Atlantic)')
ax2.plot(merge_df['ReleaseDate'],merge_df['FreightBreakeven'], color='#4F41F4', linewidth=2, label='US Arb [M+1] Freight Breakeven Level')

ax2.fill_between(merge_df['ReleaseDate'], merge_df['USDperday'], merge_df['FreightBreakeven'], \
                where=merge_df['USDperday']>merge_df['FreightBreakeven'], facecolor='red', interpolate=True, alpha=0.05)

ax2.fill_between(merge_df['ReleaseDate'], merge_df['USDperday'], merge_df['FreightBreakeven'], \
                where=merge_df['USDperday']<merge_df['FreightBreakeven'], facecolor='green', interpolate=True, alpha=0.05)

ax2.set_xlim(datetime.datetime.today() - datetime.timedelta(days=380), datetime.datetime.today())
ax2.set_ylim(-100000, 120000)

plt.title('Spark30S (Atlantic) vs. US Arb [M+1] Freight Breakeven Level')

sns.despine(left=True, bottom=True)
plt.grid(True)
plt.legend()